# DATA37711 Homework 2 Question 2

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

import copy
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler


## (a)

In [5]:
# Load data
df = pd.read_csv(
    "retail.csv",
    dtype={
        "StockCode": "string",
        "CustomerID": "string"
    }
)

categorical_cols = ["StockCode", "CustomerID"]

for col in categorical_cols:
    df[col] = df[col].astype("category")


# Count unique values
print("Feature cardinality:")
print(df[categorical_cols].nunique())

print(
    "\nTotal one-hot dimensions:",
    df[categorical_cols].nunique().sum()
)


# Define features and target
X = df[categorical_cols].copy()
y = df["Quantity"].astype(np.float64)


# Split: 70% training / 20% validation / 10% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1 / 3,
    random_state=42
)


# Reset indices for subsequent steps
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


print(f"\nTraining samples:   {len(y_train)}")
print(f"Validation samples: {len(y_val)}")
print(f"Test samples:       {len(y_test)}")

Feature cardinality:
StockCode     3665
CustomerID    4339
dtype: int64

Total one-hot dimensions: 8004

Training samples:   278546
Validation samples: 79585
Test samples:       39793


## (b)

In [6]:
# 1. Define target encoding
categorical_cols = ["StockCode", "CustomerID"]


def fit_target_mapping(X_fit, y_fit):
    """Calculate category means using only the supplied data."""
    data = X_fit.copy()
    data["Quantity"] = np.asarray(y_fit)

    mappings = {}

    for col in categorical_cols:
        mappings[col] = (
            data.groupby(col, observed=True)["Quantity"].mean()
        )

    global_mean = float(np.mean(y_fit))

    return mappings, global_mean


def apply_target_mapping(X_new, mappings, global_mean):
    """Replace categories with means; use global mean if unseen."""
    encoded = pd.DataFrame(index=X_new.index)

    for col in categorical_cols:
        encoded[col] = (
            X_new[col]
            .astype("object")
            .map(mappings[col])
            .fillna(global_mean)
            .astype(float)
        )

    return encoded.to_numpy(dtype=np.float64)


# 2. Out-of-fold target encoding for training observations
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X_train_te = np.zeros(
    (len(X_train), len(categorical_cols)),
    dtype=np.float64
)

for fit_idx, holdout_idx in kf.split(X_train):
    mappings, global_mean = fit_target_mapping(
        X_train.iloc[fit_idx],
        y_train.iloc[fit_idx]
    )

    X_train_te[holdout_idx] = apply_target_mapping(
        X_train.iloc[holdout_idx],
        mappings,
        global_mean
    )



# 3. Encode validation/test using the full training set
te_mappings, train_global_mean = fit_target_mapping(
    X_train, y_train
)

X_val_te = apply_target_mapping(
    X_val, te_mappings, train_global_mean
)

X_test_te = apply_target_mapping(
    X_test, te_mappings, train_global_mean
)


# 4. Transform the inputs and target
# Encoding uses means of ORIGINAL Quantity.
# Log-transform the resulting positive features to reduce skewness.
X_train_te_log = np.log1p(X_train_te)
X_val_te_log = np.log1p(X_val_te)
X_test_te_log = np.log1p(X_test_te)

z_train = np.log1p(y_train.to_numpy())
z_val = np.log1p(y_val.to_numpy())
z_test = np.log1p(y_test.to_numpy())


# 5. Fit linear regression using the encoded features
te_model = LinearRegression()
te_model.fit(X_train_te_log, z_train)


# 6. Evaluate on validation and test sets
# Clip log predictions at zero so predicted quantities are nonnegative.
z_val_pred_te = np.maximum(
    te_model.predict(X_val_te_log), 0.0
)

z_test_pred_te = np.maximum(
    te_model.predict(X_test_te_log), 0.0
)

# Convert predictions back to quantities
y_test_pred_te = np.expm1(z_test_pred_te)

val_rmsle_te = np.sqrt(
    mean_squared_error(z_val, z_val_pred_te)
)

test_rmsle_te = np.sqrt(
    mean_squared_error(z_test, z_test_pred_te)
)

test_mae_te = mean_absolute_error(
    y_test, y_test_pred_te
)

print("Model: Target encoding + linear regression")
print("Training loss: MSE on log(1 + Quantity)")
print(f"Validation RMSLE: {val_rmsle_te:.4f}")
print(f"Test RMSLE:       {test_rmsle_te:.4f}")
print(f"Test MAE:         {test_mae_te:.4f}")

Model: Target encoding + linear regression
Training loss: MSE on log(1 + Quantity)
Validation RMSLE: 0.6400
Test RMSLE:       0.6407
Test MAE:         7.0943


## (c)

In [8]:
# 1. Reproducibility
np.random.seed(42)
torch.manual_seed(42)


# 2. Build category mappings using TRAINING observations only
# Index 0 is reserved for unknown categories.
stock_to_idx = {
    value: idx + 1
    for idx, value in enumerate(X_train["StockCode"].unique())
}

customer_to_idx = {
    value: idx + 1
    for idx, value in enumerate(X_train["CustomerID"].unique())
}


def encode_category_indices(X_data):
    stock_idx = (
        X_data["StockCode"]
        .astype("object")
        .map(stock_to_idx)
        .fillna(0)
        .to_numpy(dtype=np.int64)
    )

    customer_idx = (
        X_data["CustomerID"]
        .astype("object")
        .map(customer_to_idx)
        .fillna(0)
        .to_numpy(dtype=np.int64)
    )

    return np.column_stack([stock_idx, customer_idx])


X_train_idx = encode_category_indices(X_train)
X_val_idx = encode_category_indices(X_val)
X_test_idx = encode_category_indices(X_test)


# 3. Create datasets and loaders
def make_embedding_dataset(X_idx, z):
    return TensorDataset(
        torch.from_numpy(X_idx),
        torch.tensor(np.asarray(z), dtype=torch.float32)
    )


train_emb_data = make_embedding_dataset(X_train_idx, z_train)
val_emb_data = make_embedding_dataset(X_val_idx, z_val)
test_emb_data = make_embedding_dataset(X_test_idx, z_test)

train_emb_loader = DataLoader(
    train_emb_data, batch_size=512, shuffle=True
)

val_emb_loader = DataLoader(
    val_emb_data, batch_size=2048, shuffle=False
)

test_emb_loader = DataLoader(
    test_emb_data, batch_size=2048, shuffle=False
)


# 4. Define embedding network
class EmbeddingRegressor(nn.Module):
    def __init__(self, n_stocks, n_customers, embedding_dim=16):
        super().__init__()

        self.stock_embedding = nn.Embedding(
            n_stocks, embedding_dim
        )

        self.customer_embedding = nn.Embedding(
            n_customers, embedding_dim
        )

        self.network = nn.Sequential(
            nn.Linear(2 * embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        stock_vector = self.stock_embedding(x[:, 0])
        customer_vector = self.customer_embedding(x[:, 1])

        combined = torch.cat(
            [stock_vector, customer_vector], dim=1
        )

        return self.network(combined).squeeze(1)


embedding_model = EmbeddingRegressor(
    n_stocks=len(stock_to_idx) + 1,
    n_customers=len(customer_to_idx) + 1,
    embedding_dim=16
)

criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    embedding_model.parameters(),
    lr=0.01
)


# 5. Train with early stopping
max_epochs = 200
patience = 15

best_val_mse_emb = float("inf")
best_state_emb = None
best_epoch_emb = None
wait = 0

for epoch in range(1, max_epochs + 1):
    embedding_model.train()
    train_total = 0.0

    for xb, yb in train_emb_loader:
        optimizer.zero_grad()

        # Replace 1% of category indices with 0 during training.
        # This trains the unknown-category embeddings.
        xb_input = xb.clone()
        unknown_mask = torch.rand(xb_input.shape) < 0.01
        xb_input[unknown_mask] = 0

        predictions = embedding_model(xb_input)
        loss = criterion(predictions, yb)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                "Nonfinite loss. Try a smaller learning rate."
            )

        loss.backward()

        nn.utils.clip_grad_norm_(
            embedding_model.parameters(), max_norm=5.0
        )

        optimizer.step()

        train_total += loss.item() * len(yb)

    train_mse = train_total / len(train_emb_data)

    # Validation without random category replacement
    embedding_model.eval()
    val_total = 0.0

    with torch.no_grad():
        for xb, yb in val_emb_loader:
            predictions = embedding_model(xb)
            loss = criterion(predictions, yb)

            val_total += loss.item() * len(yb)

    val_mse = val_total / len(val_emb_data)

    if not np.isfinite(val_mse):
        raise RuntimeError("Nonfinite validation loss.")

    if val_mse < best_val_mse_emb:
        best_val_mse_emb = val_mse
        best_state_emb = copy.deepcopy(
            embedding_model.state_dict()
        )
        best_epoch_emb = epoch
        wait = 0
    else:
        wait += 1

    # Print periodically to keep output manageable
    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train log-MSE: {train_mse:.4f} | "
            f"Validation log-MSE: {val_mse:.4f}"
        )

    if wait >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

total_epochs_emb = epoch

# Restore the best epoch
embedding_model.load_state_dict(best_state_emb)
embedding_model.eval()


# 6. Predict using the selected model
def predict_embedding(model, loader):
    predictions = []

    model.eval()

    with torch.no_grad():
        for xb, _ in loader:
            predictions.append(model(xb).numpy())

    return np.maximum(np.concatenate(predictions), 0.0)


z_val_pred_emb = predict_embedding(
    embedding_model, val_emb_loader
)

z_test_pred_emb = predict_embedding(
    embedding_model, test_emb_loader
)

y_test_pred_emb = np.expm1(
    z_test_pred_emb.astype(np.float64)
)


# 7. Evaluate using the same metrics as part (b)
val_rmsle_emb = np.sqrt(
    mean_squared_error(z_val, z_val_pred_emb)
)

test_rmsle_emb = np.sqrt(
    mean_squared_error(z_test, z_test_pred_emb)
)

test_mae_emb = mean_absolute_error(
    y_test, y_test_pred_emb
)


# 8. Print results and compare with target encoding
print(f"\nTotal epochs trained: {total_epochs_emb}")
print(f"Best epoch: {best_epoch_emb}")
print(f"Best validation log-MSE: {best_val_mse_emb:.4f}")

comparison_bc = pd.DataFrame({
    "Model": [
        "Target encoding + linear regression",
        "Embedding neural network"
    ],
    "Validation RMSLE": [
        val_rmsle_te,
        val_rmsle_emb
    ],
    "Test RMSLE": [
        test_rmsle_te,
        test_rmsle_emb
    ],
    "Test MAE": [
        test_mae_te,
        test_mae_emb
    ]
})

print("\n" + comparison_bc.round(4).to_string(index=False))

Epoch 001 | Train log-MSE: 1.0903 | Validation log-MSE: 0.9957
Epoch 005 | Train log-MSE: 0.9503 | Validation log-MSE: 0.9474
Epoch 010 | Train log-MSE: 0.9007 | Validation log-MSE: 0.9009
Epoch 015 | Train log-MSE: 0.8649 | Validation log-MSE: 0.8694
Epoch 020 | Train log-MSE: 0.8323 | Validation log-MSE: 0.8383
Epoch 025 | Train log-MSE: 0.8000 | Validation log-MSE: 0.8067
Epoch 030 | Train log-MSE: 0.7668 | Validation log-MSE: 0.7765
Epoch 035 | Train log-MSE: 0.7347 | Validation log-MSE: 0.7486
Epoch 040 | Train log-MSE: 0.7087 | Validation log-MSE: 0.7211
Epoch 045 | Train log-MSE: 0.6848 | Validation log-MSE: 0.7016
Epoch 050 | Train log-MSE: 0.6650 | Validation log-MSE: 0.6852
Epoch 055 | Train log-MSE: 0.6471 | Validation log-MSE: 0.6830
Epoch 060 | Train log-MSE: 0.6318 | Validation log-MSE: 0.6608
Epoch 065 | Train log-MSE: 0.6184 | Validation log-MSE: 0.6384
Epoch 070 | Train log-MSE: 0.6068 | Validation log-MSE: 0.6484
Epoch 075 | Train log-MSE: 0.5962 | Validation log-MSE:

## (d)

In [10]:
# 1. Prepare target-encoded features
# Use the leakage-free features created in part (b).
# Fit the scaler on training features only.
te_scaler = StandardScaler()

X_train_te_scaled = te_scaler.fit_transform(
    X_train_te_log
).astype(np.float32)

X_val_te_scaled = te_scaler.transform(
    X_val_te_log
).astype(np.float32)

X_test_te_scaled = te_scaler.transform(
    X_test_te_log
).astype(np.float32)


# 2. Create datasets with category indices AND target encoding
def make_combined_dataset(X_idx, X_te, z):
    return TensorDataset(
        torch.from_numpy(X_idx),
        torch.from_numpy(X_te),
        torch.tensor(np.asarray(z), dtype=torch.float32)
    )


train_comb_data = make_combined_dataset(
    X_train_idx, X_train_te_scaled, z_train
)

val_comb_data = make_combined_dataset(
    X_val_idx, X_val_te_scaled, z_val
)

test_comb_data = make_combined_dataset(
    X_test_idx, X_test_te_scaled, z_test
)

# Reset seeds before initializing and training the new model
np.random.seed(42)
torch.manual_seed(42)

train_comb_loader = DataLoader(
    train_comb_data, batch_size=512, shuffle=True
)

val_comb_loader = DataLoader(
    val_comb_data, batch_size=2048, shuffle=False
)

test_comb_loader = DataLoader(
    test_comb_data, batch_size=2048, shuffle=False
)


# 3. Define the combined network
class CombinedRegressor(nn.Module):
    def __init__(self, n_stocks, n_customers, embedding_dim=16):
        super().__init__()

        self.stock_embedding = nn.Embedding(
            n_stocks, embedding_dim
        )

        self.customer_embedding = nn.Embedding(
            n_customers, embedding_dim
        )

        self.network = nn.Sequential(
            nn.Linear(2 * embedding_dim + 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x_idx, x_te):
        stock_vector = self.stock_embedding(x_idx[:, 0])
        customer_vector = self.customer_embedding(x_idx[:, 1])

        combined = torch.cat(
            [stock_vector, customer_vector, x_te],
            dim=1
        )

        return self.network(combined).squeeze(1)


combined_model = CombinedRegressor(
    n_stocks=len(stock_to_idx) + 1,
    n_customers=len(customer_to_idx) + 1,
    embedding_dim=16
)

criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    combined_model.parameters(),
    lr=0.01
)


# 4. Train with early stopping
max_epochs = 200
patience = 15

best_val_mse_comb = float("inf")
best_state_comb = None
best_epoch_comb = None
wait = 0

for epoch in range(1, max_epochs + 1):
    combined_model.train()
    train_total = 0.0

    for xb_idx, xb_te, yb in train_comb_loader:
        optimizer.zero_grad()

        # Train unknown-category embeddings, as in part (c).
        xb_input = xb_idx.clone()
        unknown_mask = torch.rand(xb_input.shape) < 0.01
        xb_input[unknown_mask] = 0

        predictions = combined_model(xb_input, xb_te)
        loss = criterion(predictions, yb)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                "Nonfinite loss. Try a smaller learning rate."
            )

        loss.backward()

        nn.utils.clip_grad_norm_(
            combined_model.parameters(), max_norm=5.0
        )

        optimizer.step()

        train_total += loss.item() * len(yb)

    train_mse = train_total / len(train_comb_data)

    # Validation
    combined_model.eval()
    val_total = 0.0

    with torch.no_grad():
        for xb_idx, xb_te, yb in val_comb_loader:
            predictions = combined_model(xb_idx, xb_te)
            loss = criterion(predictions, yb)

            val_total += loss.item() * len(yb)

    val_mse = val_total / len(val_comb_data)

    if not np.isfinite(val_mse):
        raise RuntimeError("Nonfinite validation loss.")

    if val_mse < best_val_mse_comb:
        best_val_mse_comb = val_mse
        best_state_comb = copy.deepcopy(
            combined_model.state_dict()
        )
        best_epoch_comb = epoch
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train log-MSE: {train_mse:.4f} | "
            f"Validation log-MSE: {val_mse:.4f}"
        )

    if wait >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

total_epochs_comb = epoch

# Restore the parameters from the best validation epoch
combined_model.load_state_dict(best_state_comb)
combined_model.eval()


# 5. Predict using the selected model
def predict_combined(model, loader):
    predictions = []

    model.eval()

    with torch.no_grad():
        for xb_idx, xb_te, _ in loader:
            predictions.append(
                model(xb_idx, xb_te).numpy()
            )

    return np.maximum(np.concatenate(predictions), 0.0)


z_val_pred_comb = predict_combined(
    combined_model, val_comb_loader
)

z_test_pred_comb = predict_combined(
    combined_model, test_comb_loader
)

y_test_pred_comb = np.expm1(
    z_test_pred_comb.astype(np.float64)
)


# 6. Evaluate
val_rmsle_comb = np.sqrt(
    mean_squared_error(z_val, z_val_pred_comb)
)

test_rmsle_comb = np.sqrt(
    mean_squared_error(z_test, z_test_pred_comb)
)

test_mae_comb = mean_absolute_error(
    y_test, y_test_pred_comb
)


# 7. Compare all three models
print(f"\nTotal epochs trained: {total_epochs_comb}")
print(f"Best epoch: {best_epoch_comb}")
print(f"Best validation log-MSE: {best_val_mse_comb:.4f}")

comparison_bcd = pd.DataFrame({
    "Model": [
        "Target encoding + linear regression",
        "Embedding neural network",
        "Target encoding + embedding network"
    ],
    "Validation RMSLE": [
        val_rmsle_te,
        val_rmsle_emb,
        val_rmsle_comb
    ],
    "Test RMSLE": [
        test_rmsle_te,
        test_rmsle_emb,
        test_rmsle_comb
    ],
    "Test MAE": [
        test_mae_te,
        test_mae_emb,
        test_mae_comb
    ]
})

print("\n" + comparison_bcd.round(4).to_string(index=False))

Epoch 001 | Train log-MSE: 0.6169 | Validation log-MSE: 0.4337
Epoch 005 | Train log-MSE: 0.4067 | Validation log-MSE: 0.4005
Epoch 010 | Train log-MSE: 0.3957 | Validation log-MSE: 0.3947
Epoch 015 | Train log-MSE: 0.3918 | Validation log-MSE: 0.3883
Epoch 020 | Train log-MSE: 0.3894 | Validation log-MSE: 0.3868
Epoch 025 | Train log-MSE: 0.3877 | Validation log-MSE: 0.3855
Epoch 030 | Train log-MSE: 0.3862 | Validation log-MSE: 0.3892
Epoch 035 | Train log-MSE: 0.3849 | Validation log-MSE: 0.3854
Epoch 040 | Train log-MSE: 0.3838 | Validation log-MSE: 0.3820
Epoch 045 | Train log-MSE: 0.3828 | Validation log-MSE: 0.3807
Epoch 050 | Train log-MSE: 0.3818 | Validation log-MSE: 0.3802
Epoch 055 | Train log-MSE: 0.3809 | Validation log-MSE: 0.3791
Epoch 060 | Train log-MSE: 0.3799 | Validation log-MSE: 0.3844
Epoch 065 | Train log-MSE: 0.3789 | Validation log-MSE: 0.3782
Epoch 070 | Train log-MSE: 0.3781 | Validation log-MSE: 0.3770
Epoch 075 | Train log-MSE: 0.3771 | Validation log-MSE: